# Universal EDA Boilerplate — understand any dataframe in minutes

**Use in any circumstance:** point it at a file/URL, pass an in-memory `DF`, or pick a vendored `SAMPLE`. Set the target/date once, run all, read the `→` insight lines out loud.

| What you set | Options |
|---|---|
| `DATA_PATH` | local `.csv`/`.parquet` path **or** `http(s)` URL (csv-over-URL works, parquet-over-URL does not) |
| `DF` | an already-loaded `pd.DataFrame` — takes precedence over `DATA_PATH` |
| `SAMPLE` | `"mpg"`, `"penguins"`, `"iris"`, `"diamonds"`, `"births"`, `"sunspots"` (vendored `data/*.csv`, runs offline) or `None` |
| `TARGET_COL` | label column, or `None` = profile-only (no supervised sections) |
| `DATE_COL` | datetime column, or `None` = auto-detect |
| `ID_COLS` | known id columns to exclude from modelling + leakage checks |

Sections: 0 setup → 1 load → 2 overview → 3 missingness → 4 numerics → 5 categoricals → 6 datetimes → 7 target (auto: classification / regression / time-aware) → 8 correlations & leakage → 9 bivariate vs target → 10 quality gates + export.
Charts: **matplotlib/seaborn** for quick scans, **plotly** for the correlation heatmap + time trend (interactive).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

try:
    import plotly.express as px
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False
    print("plotly not available — heatmap/trend fall back to matplotlib.")

print("pandas", pd.__version__, "| plotly:", HAS_PLOTLY)
print("ready — edit Cell 2, then Run All")

## 1 — Configuration (the ONLY cell you must edit)

In [ ]:
# ---- Flexible input: DF > DATA_PATH > SAMPLE > synthetic fallback ----
DF = None                  # e.g. DF = my_df  (in-memory takes precedence)
DATA_PATH = None           # e.g. "data/mpg.csv" or "https://.../file.csv" (.csv/.parquet)
SAMPLE = "mpg"              # "mpg"|"penguins"|"iris"|"diamonds"|"births"|"sunspots"|None
TARGET_COL = "mpg"          # label column, or None for profile-only
DATE_COL = None             # datetime col, or None for auto-detect
ID_COLS = []                # e.g. ["id", "name"]

SAMPLE_FILES = {
    "mpg": ("data/mpg.csv", "mpg"),
    "penguins": ("data/penguins.csv", "species"),
    "iris": ("data/iris.csv", None),
    "diamonds": ("data/diamonds.csv", "price"),
    "births": ("data/daily-female-births.csv", "Births"),
    "sunspots": ("data/monthly-sunspots.csv", "Sunspots"),
}
RANDOM_STATE = 42

## 2 — Universal loader + first look

In [ ]:
def load_table(path: str) -> pd.DataFrame:
    p = str(path)
    is_url = p.startswith(("http://", "https://"))
    ext = p.split("?")[0].lower().rsplit(".", 1)[-1] if "." in p.split("?")[0] else ""
    if ext == "parquet":
        assert not is_url, "parquet-over-URL needs fsspec — download it locally first"
        return pd.read_parquet(p)
    return pd.read_csv(p)  # csv works for local paths AND urls

def _repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "mpg.csv").exists():
            return p
    return Path(".")

df = None
source_note = ""
if DF is not None:
    df = DF.copy()
    source_note = "in-memory DF"
elif DATA_PATH:
    df = load_table(DATA_PATH)
    source_note = f"DATA_PATH={DATA_PATH}"
elif SAMPLE in SAMPLE_FILES:
    rel, default_target = SAMPLE_FILES[SAMPLE]
    fp = _repo_root() / rel
    try:
        df = load_table(str(fp))
        source_note = f"SAMPLE={SAMPLE} ({fp})"
        if TARGET_COL == "mpg" and SAMPLE != "mpg" and default_target:
            TARGET_COL = default_target  # sensible default per sample
            print(f"TARGET_COL auto-set to '{TARGET_COL}' for sample '{SAMPLE}'")
    except FileNotFoundError:
        df = None
if df is None:  # synthetic fallback so every later cell still runs
    print("⚠ no data found — using synthetic demo (set DF / DATA_PATH / SAMPLE for real data).")
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1000
    df = pd.DataFrame({
        "num_a": rng.normal(0, 1, n),
        "num_b": rng.exponential(1.0, n),
        "cat_c": rng.choice(["a", "b", "c", None], n, p=[.4, .3, .2, .1]),
        "date": pd.date_range("2024-01-01", periods=n, freq="D"),
    })
    df["target"] = ((df["num_a"] + rng.normal(0, .5, n)) > 0).astype(int)
    df.loc[rng.choice(n, 50, replace=False), "num_b"] = np.nan
    if TARGET_COL is None:
        TARGET_COL = "target"
    if DATE_COL is None:
        DATE_COL = "date"
    source_note = "synthetic fallback"

# normalize target/date pointers
if TARGET_COL is not None and TARGET_COL not in df.columns:
    print(f"⚠ TARGET_COL='{TARGET_COL}' not in columns — profile-only mode.")
    TARGET_COL = None
if DATE_COL is not None and DATE_COL not in df.columns:
    print(f"⚠ DATE_COL='{DATE_COL}' not in columns — ignoring.")
    DATE_COL = None

print(f"source: {source_note} | shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head())
print("\ndtypes:"); print(df.dtypes)
print(f"\nTARGET_COL={TARGET_COL} | DATE_COL={DATE_COL} | ID_COLS={ID_COLS}")

## 3 — Overview: shape, memory, dtypes, duplicates, constants, ID-like

In [ ]:
n_rows, n_cols = df.shape
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"rows={n_rows:,} cols={n_cols} memory={mem_mb:.1f} MB | source: {source_note}")
print(f"duplicated rows: {df.duplicated().sum():,} ({df.duplicated().mean():.2%})")

prof = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
    "n_unique_nonnull": df.nunique(dropna=True),
})
prof["is_constant"] = prof["n_unique"] <= 1
prof["is_id_like"] = (prof["n_unique"] >= 0.95 * max(len(df), 1)) & (len(df) > 10)
display(prof.sort_values(["is_id_like", "pct_missing"], ascending=[False, False]))

const_cols = prof.index[prof["is_constant"]].tolist()
id_like = prof.index[prof["is_id_like"]].tolist() + [c for c in ID_COLS if c in df.columns]
print(f"\nconstant cols: {const_cols or '—'}")
print(f"ID-like cols (unique/row, drop from features): {list(dict.fromkeys(id_like)) or '—'}")
num_cols_all = df.select_dtypes(include='number').columns.tolist()
cat_cols_all = df.select_dtypes(exclude='number').columns.tolist()
print(f"numeric: {len(num_cols_all)} | non-numeric: {len(cat_cols_all)}")
print("→ Say: rows/cols/memory, dup rate, and which cols are constants/IDs before any modelling.")

## 4 — Missingness (counts + pattern + is-missing-predictive?)

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"columns with missing: {len(miss)}/{df.shape[1]} | total cells missing: {df.isna().sum().sum():,} ({df.isna().mean().mean():.2%})")
if len(miss):
    display(pd.DataFrame({"pct_missing": (miss * 100).round(2),
                           "n_missing": df.isna().sum()[miss.index]}))
    plt.figure(figsize=(10, max(2.5, .45 * len(miss))))
    miss.plot(kind="barh")
    plt.xlabel("fraction missing"); plt.title("Missingness by column"); plt.tight_layout(); plt.show()
else:
    print("No missing values — say so explicitly, then move on.")

# Is missingness itself predictive of the target? (quick MNAR check)
if TARGET_COL in (df.columns if TARGET_COL else []) and len(miss):
    y = df[TARGET_COL]
    print("\nP(target | col missing) vs present — big gaps = INFORMATIVE missingness:")
    for c in miss.index[:5]:
        m = df[c].isna()
        if m.mean() in (0, 1):
            continue
        if pd.api.types.is_numeric_dtype(y):
            print(f"  {c}: mean(target|missing)={y[m].mean():.3f} vs present={y[~m].mean():.3f}")
        else:
            ct = pd.crosstab(m, y, normalize="index").round(3)
            print(f"  {c}:\n{ct.to_string()}")
    print("→ If informative: keep a missing-indicator + impute; if MCAR-looking: plain impute is fine.")

## 5 — Numeric features: shape, skew, outliers, zeros

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET_COL]
print(f"{len(num_cols)} numeric feature cols" + (f" (target '{TARGET_COL}' excluded)" if TARGET_COL else ""))
if num_cols:
    desc = df[num_cols].describe().T
    desc["skew"] = df[num_cols].skew()
    desc["kurt"] = df[num_cols].kurt()
    desc["zeros_pct"] = (df[num_cols] == 0).mean().round(4) * 100
    display(desc.round(3))

    k = min(len(num_cols), 6)
    fig, axes = plt.subplots(1, k, figsize=(4 * k, 3))
    for ax, c in zip(np.atleast_1d(axes), num_cols[:k]):
        ax.hist(df[c].dropna(), bins=30); ax.set_title(c)
    plt.suptitle(f"Histograms (first {k} numeric)"); plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, k, figsize=(4 * k, 3))
    for ax, c in zip(np.atleast_1d(axes), num_cols[:k]):
        ax.boxplot(df[c].dropna(), vert=True); ax.set_title(c)
    plt.suptitle(f"Boxplots (first {k} numeric)"); plt.tight_layout(); plt.show()

    out = {}
    for c in num_cols:
        s = df[c].dropna()
        if len(s) == 0:
            continue
        q1, q3 = s.quantile(.25), s.quantile(.75)
        iqr = q3 - q1
        if iqr > 0:
            out[c] = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()
    out = pd.Series(out).sort_values(ascending=False)
    if len(out):
        print("\noutlier share (IQR rule):"); print(out.round(4).to_string())
    print("\n→ Say: skew (|skew|>1 → log-transform?), zeros (zero-inflation? MAPE unstable?), outliers (entry error vs heavy tail → robust scaler / trees).")
else:
    print("No numeric feature columns.")

## 6 — Categorical features: cardinality, top values, rare levels

In [ ]:
cat_cols = [c for c in df.select_dtypes(include=["object", "category", "bool"]).columns if c != TARGET_COL]
for c in df.select_dtypes(include="number").columns:  # low-card numerics are often codes
    if c != TARGET_COL and df[c].nunique(dropna=True) <= 10 and c not in cat_cols:
        cat_cols.append(c)
print(f"{len(cat_cols)} categorical-ish cols")
for c in cat_cols[:12]:
    vc = df[c].value_counts(dropna=False).head(8)
    share = (vc / len(df)).round(4)
    print(f"\n{c} — {df[c].nunique(dropna=False)} unique (top 8 share):")
    print(share.to_string())
    rare = (df[c].value_counts(normalize=True, dropna=False) < 0.01).sum()
    if df[c].nunique(dropna=False) == len(df):
        print("  ⚠ unique per row → ID-like, drop from features.")
    elif df[c].nunique(dropna=False) > 50:
        print("  ⚠ high cardinality → target/frequency encoding or hashing, not one-hot.")
    elif rare:
        print(f"  → {rare} rare level(s) <1%: group into 'Other' to avoid overfitting.")
if not cat_cols:
    print("No categorical columns.")
print("\n→ Say: cardinality, dominant levels, and the encoding plan (one-hot vs target/frequency).")

## 7 — Datetime features (auto-detected unless DATE_COL set)

In [ ]:
dt_cols = []
if DATE_COL and DATE_COL in df.columns:
    dt_cols = [DATE_COL]
else:  # auto-detect: already-datetime dtypes + object cols that mostly parse as dates
    dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
    for c in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[c].dropna().head(200), errors="coerce")
            if len(parsed.dropna()) / max(len(df[c].dropna().head(200)), 1) > 0.9:
                dt_cols.append(c)
        except Exception:
            pass
print(f"datetime cols: {dt_cols or '—'}")
TIME_COL = dt_cols[0] if dt_cols else None

if TIME_COL:
    t = pd.to_datetime(df[TIME_COL], errors="coerce")
    print(f"using '{TIME_COL}': range {t.min()} → {t.max()} | NaT: {t.isna().sum()}")
    dd = pd.DataFrame({"t": t}).dropna().sort_values("t")
    if len(dd) > 1:
        gaps = dd["t"].diff().dropna()
        print(f"median gap: {gaps.median()} | max gap: {gaps.max()}")
        inferred = pd.infer_freq(dd["t"].head(50))
        print(f"inferred freq: {inferred}")
        # resampled trend
        rule = "ME" if (gaps.median() >= pd.Timedelta(days=20)) else "D"
        trend = dd.set_index("t").assign(n=1)["n"].resample(rule).sum()
        if HAS_PLOTLY:
            fig = px.line(x=trend.index, y=trend.values, title=f"Row volume over time ({rule})")
            fig.show()
        else:
            plt.figure(); plt.plot(trend.index, trend.values); plt.title(f"Row volume over time ({rule})")
            plt.tight_layout(); plt.show()
        if TARGET_COL and pd.api.types.is_numeric_dtype(df[TARGET_COL]):
            yy = pd.DataFrame({"t": t, "y": pd.to_numeric(df[TARGET_COL], errors="coerce")}).dropna().sort_values("t")
            if len(yy):
                ym = yy.set_index("t")["y"].resample(rule).mean()
                plt.figure(); plt.plot(ym.index, ym.values); plt.title(f"Mean(target) over time ({rule})")
                plt.tight_layout(); plt.show()
                print("→ Trending target + trees? say 'trees can't extrapolate trend — add trend feature / try boosting'.")
    print("→ Say: coverage, gaps, granularity, and whether train/test must split chronologically.")
else:
    print("No datetime column — skipping time/drift checks (set DATE_COL to force one).")

## 8 — Target analysis (auto: classification vs regression, time-aware if datetime present)

In [ ]:
TASK = None
if TARGET_COL not in (df.columns if TARGET_COL else []):
    print("No TARGET_COL — profile-only mode, skipping supervised sections.")
else:
    y = df[TARGET_COL]
    nuniq = y.nunique(dropna=True)
    is_dt_target = pd.api.types.is_datetime64_any_dtype(y)
    if not is_dt_target and (pd.api.types.is_bool_dtype(y) or pd.api.types.is_object_dtype(y)
                             or isinstance(y.dtype, pd.CategoricalDtype) or nuniq <= 20):
        TASK = "classification"
    else:
        TASK = "regression"
    if TIME_COL:
        TASK += "+time"
    print(f"TASK = {TASK}  (target '{TARGET_COL}', {nuniq} unique, missing {y.isna().sum()} / {y.isna().mean():.2%})")
    print(y.describe(include="all").to_string())

    fig, ax = plt.subplots(figsize=(8, 3.5))
    if TASK.startswith("classification"):
        vc = y.value_counts(dropna=False, normalize=True).sort_index()
        vc.plot(kind="bar", ax=ax); ax.set_ylabel("share")
        ax.set_title(f"Target distribution — imbalance {vc.max() / max(vc.min(), 1e-9):.1f}:1")
        plt.tight_layout(); plt.show()
        print("\nclass balance (share):"); print(vc.round(4).to_string())
        if vc.min() < 0.10:
            print("→ MINORITY <10%: stratify splits, report PR/F1 — not accuracy alone.")
        if nuniq > 2:
            print(f"→ {nuniq}-class: weighted precision/recall/F1, confusion matrix per class.")
    else:
        ax.hist(pd.to_numeric(y, errors="coerce").dropna(), bins=50)
        ax.set_title("Target distribution (regression)"); plt.tight_layout(); plt.show()
        yn = pd.to_numeric(y, errors="coerce")
        print(f"skew={yn.skew():.2f} kurt={yn.kurt():.2f} zeros={(yn == 0).mean():.2%}")
        if abs(yn.skew()) > 1:
            print("→ |skew|>1: mention log-transform / robust loss.")
        if (yn == 0).mean() > 0.05:
            print("→ >5% zeros: zero-inflation? MAPE undefined at 0 — prefer sMAPE/MAE.")
    if TIME_COL and TASK.endswith("+time"):
        print("→ Time-aware target: split chronologically, lag/rolling features on shift(1), no shuffling.")

## 9 — Correlations, multicollinearity & leakage suspects (plotly heatmap)

In [ ]:
num_feat = [c for c in df.select_dtypes(include="number").columns if c != TARGET_COL]
leak_suspects, high_pairs = [], []
if num_feat and TARGET_COL in (df.columns if TARGET_COL else []):
    tmp = df[num_feat + [TARGET_COL]].copy()
    if TASK and TASK.startswith("classification"):
        tmp[TARGET_COL] = pd.factorize(tmp[TARGET_COL])[0]
    tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors="coerce")
    corr_t = tmp.corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL, errors="ignore").sort_values(key=abs, ascending=False)
    print("top |corr| with target:"); print(corr_t.head(10).round(3).to_string())
    leak_suspects = corr_t[corr_t.abs() > 0.9].index.tolist()
    if leak_suspects:
        print(f"\n⚠ LEAKAGE SUSPECTS (|r|>0.9): {leak_suspects} — 'computed after the outcome? check timestamps'.")
    # inter-feature multicollinearity
    C = tmp[num_feat[:15]].corr()
    for i in range(len(C.columns)):
        for j in range(i + 1, len(C.columns)):
            if abs(C.iloc[i, j]) > 0.8:
                high_pairs.append((C.columns[i], C.columns[j], round(float(C.iloc[i, j]), 3)))
    if high_pairs:
        print(f"\nmulticollinear pairs (|r|>0.8): {high_pairs[:10]} → 'drop one / regularize / PCA'.")
    if HAS_PLOTLY and len(C) > 1:
        fig = px.imshow(C, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r",
                        title="Feature correlation (multicollinearity check)")
        fig.show()
    elif len(C) > 1:
        plt.figure(figsize=(8, 6)); sns.heatmap(C, cmap="coolwarm", center=0); plt.title("Feature correlation")
        plt.tight_layout(); plt.show()
elif num_feat:
    print("Profile-only: inter-feature correlation (no target).")
    C = df[num_feat[:15]].corr()
    if HAS_PLOTLY and len(C) > 1:
        fig = px.imshow(C, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r",
                        title="Feature correlation")
        fig.show()
else:
    print("No numeric features — skipping correlation section.")

print(f"\nstructural checks → constant: {[c for c in df.columns if df[c].nunique(dropna=False) <= 1]}")
print(f"ID-like: {[c for c in df.columns if df[c].nunique(dropna=False) == len(df)]} + {ID_COLS}")
print("→ Ask: 'which columns would NOT be known at prediction time?' — that is your leakage list.")

## 10 — Bivariate vs target: numeric + categorical signals

In [ ]:
if TARGET_COL not in (df.columns if TARGET_COL else []):
    print("Profile-only — skipping bivariate section.")
else:
    y = df[TARGET_COL]
    if TASK and TASK.startswith("classification"):
        print("means of numerics by class (first 8):")
        display(df[num_feat[:8] + [TARGET_COL]].groupby(TARGET_COL, dropna=False).mean(numeric_only=True).round(3)
                if num_feat else pd.DataFrame())
        for c in cat_cols[:4]:
            print(f"\n{c} × target (row-normalized):")
            print(pd.crosstab(df[c].fillna("∅_missing"), y, normalize="index").round(3).to_string())
    else:
        yn = pd.to_numeric(y, errors="coerce")
        print("numeric vs target corr (top 8 by |r|):")
        rs = {c: pd.Series(pd.to_numeric(df[c], errors="coerce")).corr(yn) for c in num_feat}
        print(pd.Series(rs).sort_values(key=abs, ascending=False).head(8).round(3).to_string())
        for c in cat_cols[:4]:
            print(f"\nmean(target) by {c}:")
            print(df.assign(__y=yn).groupby(c, dropna=False)["__y"].agg(["mean", "count"]).sort_values("mean").round(3).head(10).to_string())
    print("\n→ Say which 3-5 features carry the signal and which look like noise.")

## 11 — Quality gates, insight checklist + export

In [ ]:
flags = {
    "dup_pct": round(float(df.duplicated().mean()), 4),
    "cols_constant": [c for c in df.columns if df[c].nunique(dropna=False) <= 1],
    "cols_id_like": [c for c in df.columns if df[c].nunique(dropna=False) == len(df)],
    "cols_high_missing_gt30": df.columns[df.isna().mean() > 0.3].tolist(),
    "leak_suspects_corr_gt90": leak_suspects if "leak_suspects" in dir() else [],
    "multicollinear_pairs": high_pairs if "high_pairs" in dir() else [],
    "task": TASK if "TASK" in dir() else None,
}
print(json.dumps(flags, indent=2, default=str))
print("""
Checklist — say each line out loud before modelling:
[ ] splits: stratified (classification) / chronological (time) / random (regression)?
[ ] missing: informative (add indicator) or MCAR (impute: median / most_frequent)?
[ ] encoding: one-hot (low-card) vs target/frequency (high-card) vs drop (IDs)?
[ ] scaling: standard vs robust (outliers/skew)?
[ ] target: transform (log?), metric (F1/PR vs accuracy? MAE/sMAPE vs MAPE?)?
[ ] leakage: every feature knowable at prediction time?
""")
try:
    from ml_boilerplate.eda import profile_dataframe
    prof_export = profile_dataframe(df)
    _root = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q / "data" / "mpg.csv").exists()), Path.cwd())
    (_root / "artifacts").mkdir(exist_ok=True)
    (_root / "artifacts" / "eda_universal_profile.json").write_text(json.dumps(prof_export, indent=2, default=str))
    print(f"saved {(_root / 'artifacts' / 'eda_universal_profile.json')} via ml_boilerplate.eda.profile_dataframe")
except Exception as e:
    print(f"(export skipped: {e})")

### Next steps
- Modelling: `notebooks/library_usage.ipynb` (à la carte recipes) or the CLI (`--config … train`).
- Interview narration: `notebooks/interview_boilerplate.ipynb` (4-var live kit).
- This notebook stays **read-only on data** — no imputation/encoding here, only diagnosis.